# Phase 8 · Repeat chunking on a fresh T4
Run all cells. The first workload repeats the earlier experiment. The next two test different sentence lengths.

In [ ]:
import os
import shutil
import subprocess
import tempfile

smi = shutil.which("nvidia-smi")
if not smi:
    raise RuntimeError("No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell.")
subprocess.run([smi, "-L"], check=True)

%pip -q install uv
os.environ["VOICEFORGE_PHASE8_RESULTS"] = tempfile.mkdtemp(prefix="voiceforge-phase8-", dir="/content")

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi -L; then
  echo "No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell." >&2
  exit 1
fi
if ! command -v espeak-ng >/dev/null; then
  apt-get -qq update
  apt-get -qq install -y espeak-ng
fi
repo=/content/voiceforge-repeat
if [[ ! -d "$repo/.git" ]]; then
  git clone --branch voiceforge-gpu-precision https://github.com/dne-labs/Voice-inference.git "$repo"
fi
git -C "$repo" checkout --detach e86db91a5abd886aa0d02dd10441b850c07b9dbe
cd "$repo/services/inference"
uv run --locked --python 3.12 \
  --index https://download.pytorch.org/whl/cu126 \
  --with 'torch==2.8.0+cu126' --with 'kokoro==0.9.4' \
  --with 'en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl' \
  voiceforge-synthesize --device cuda --output /tmp/voiceforge-phase8-warmup.wav

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi -L; then
  echo "No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell." >&2
  exit 1
fi
cd /content/voiceforge-repeat/services/inference
HF_HUB_OFFLINE=1 OMP_NUM_THREADS=2 uv run --locked --python 3.12 \
  --index https://download.pytorch.org/whl/cu126 \
  --with 'torch==2.8.0+cu126' --with 'kokoro==0.9.4' \
  --with 'en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl' \
  python - <<'PY'
import json
import os
import sys
from pathlib import Path
import torch
sys.path.insert(0, "../../experiments/06_adaptive_chunking")
import load
from voiceforge_inference.models.kokoro import KokoroAdapter

root = Path(os.environ["VOICEFORGE_PHASE8_RESULTS"])
model = KokoroAdapter(device="cuda")
prompts = {"control": load.PROMPT, "short": "Your order is ready. Pick it up at gate five.", "narration": "When the rain finally stopped, the crew walked down the platform and waited for the evening train. A few minutes later, the station lights came on. Beyond the tracks, the last patch of sunlight disappeared behind the hills."}
report = {"gpu": torch.cuda.get_device_name(), "torch": str(torch.__version__), "precision": "fp32", "voice": "af_heart", "runner_commit": "e86db91a5abd886aa0d02dd10441b850c07b9dbe", "workloads": {}}
for name, prompt in prompts.items():
    load.PROMPT = prompt
    folder = root / name
    folder.mkdir(exist_ok=True)
    bursts = load.benchmark(model, folder)
    report["workloads"][name] = {"prompt": prompt, "bursts": bursts}
    (root / "repeat.json").write_text(json.dumps(report, indent=2))
PY

In [ ]:
from pathlib import Path
import json
import os
import shutil
from google.colab import files

results = Path(os.environ["VOICEFORGE_PHASE8_RESULTS"])
from IPython.display import Audio, display
for policy in ("paragraph", "sentence"):
    print(policy)
    display(Audio(filename=str(results / "control" / f"{policy}.wav")))
print("Saved repeat.json")
files.download(shutil.make_archive(str(results), "zip", results))
